# Canonical Transformations in Hamiltonian Mechanics

This notebook contains the programmatic verification for the **Canonical Transformations in Hamiltonian Mechanics** entry from the THEORIA dataset.

**Entry ID:** canonical_transformations  
**Required Library:** sympy 1.13.1

## Description
Canonical transformations are differentiable, invertible changes of phase-space coordinates `(q,p) -> (Q,P)` that preserve the canonical form of Hamilton's equations. Equivalently, they preserve the fundamental Poisson brackets and their Jacobian is symplectic. Locally they can be generated by functions such as `F_2(q,P,t)`, which also determines the transformed Hamiltonian through `K = H + (del F_2)/(del t)`.

## Installation
First, let's install the required library:

In [ ]:
# Install required library with exact version
!pip install sympy==1.13.1

## Programmatic Verification

The following code verifies the derivation mathematically:

In [ ]:
import sympy as sp
from sympy import Matrix, eye, zeros, diff, expand, simplify

# =====================================================
# Programmatic verification: Canonical Transformations
#
# Steps 1-2 rely on the dependency `hamiltons_equations`.
# Steps 3-9 derive the generating-function relations.
# Steps 10-12 derive the symplectic condition from bracket invariance.
# Steps 13-15 verify the fundamental Poisson brackets.
# =====================================================

# ---------------------------
# Steps 3-6: Type-1 generating function F_1(q,Q,t)
# ---------------------------
# For one degree of freedom, the action-integrand relation is
#   p*qdot - H = P*Qdot - K + dF_1/dt.
# Expanding dF_1/dt and collecting the independent velocities qdot and Qdot
# yields the Type-1 generating-function relations.
p, P = sp.symbols('p P', real=True)
H, K = sp.symbols('H K', real=True)
qdot, Qdot = sp.symbols('qdot Qdot', real=True)
F1_q, F1_Q, F1_t = sp.symbols('F1_q F1_Q F1_t', real=True)

dF1_dt = F1_q*qdot + F1_Q*Qdot + F1_t
residual_F1 = expand((p*qdot - H) - (P*Qdot - K + dF1_dt))

assert residual_F1.coeff(qdot) == p - F1_q
assert residual_F1.coeff(Qdot) == -P - F1_Q
assert simplify(residual_F1.subs({qdot: 0, Qdot: 0}) - (K - H - F1_t)) == 0

# ---------------------------
# Steps 7-9: Type-2 generating function F_2(q,P,t)
# ---------------------------
# Define F_2 = F_1 + Q*P. Using dF_1 = p*dq - P*dQ + (K-H)dt,
# we obtain dF_2 = p*dq + Q*dP + (K-H)dt.
dq, dQ, dP, dt = sp.symbols('dq dQ dP dt', real=True)
Qsym = sp.symbols('Q', real=True)

dF1 = p*dq - P*dQ + (K - H)*dt
dF2_from_F1 = expand(dF1 + Qsym*dP + P*dQ)
expected_dF2 = p*dq + Qsym*dP + (K - H)*dt
assert simplify(dF2_from_F1 - expected_dF2) == 0

F2_q, F2_P, F2_t = sp.symbols('F2_q F2_P F2_t', real=True)
dF2_direct = F2_q*dq + F2_P*dP + F2_t*dt

assert dF2_direct.coeff(dq) == F2_q
assert dF2_direct.coeff(dP) == F2_P
assert dF2_direct.coeff(dt) == F2_t

assert expected_dF2.coeff(dq) == p
assert expected_dF2.coeff(dP) == Qsym
assert expected_dF2.coeff(dt) == K - H

K_from_F2 = H + expected_dF2.coeff(dt)
assert simplify(K_from_F2 - K) == 0

# ---------------------------
# Steps 10-12: Matrix form of the Poisson bracket and symplectic condition
# ---------------------------
n = 2
I_n = eye(n)
O_n = zeros(n)
J = Matrix([[O_n, I_n], [-I_n, O_n]])

assert J.shape == (2*n, 2*n)
assert J[:n, :n] == O_n
assert J[:n, n:] == I_n
assert J[n:, :n] == -I_n
assert J[n:, n:] == O_n

M = sp.MatrixSymbol('M', 2*n, 2*n)
grad_u_eps = sp.MatrixSymbol('grad_u_eps', 2*n, 1)
grad_v_eps = sp.MatrixSymbol('grad_v_eps', 2*n, 1)

pb_eta = (M.T * grad_u_eps).T * J * (M.T * grad_v_eps)
pb_rewritten = grad_u_eps.T * M * J * M.T * grad_v_eps
assert pb_eta == pb_rewritten

pb_eps = grad_u_eps.T * J * grad_v_eps
assert pb_rewritten.subs(M * J * M.T, J) == pb_eps

# ---------------------------
# Steps 13-15: Fundamental brackets and symplectic condition in 1 DOF
# ---------------------------
# For Q = a*q + b*p and P = c*q + d*p, compute the brackets exactly.
q, p_old = sp.symbols('q p_old', real=True)
a, b, c, d = sp.symbols('a b c d', real=True)

Q_lin = a*q + b*p_old
P_lin = c*q + d*p_old

def poisson_bracket_1d(u, v):
    return simplify(diff(u, q)*diff(v, p_old) - diff(u, p_old)*diff(v, q))

pb_QP = poisson_bracket_1d(Q_lin, P_lin)
pb_QQ = poisson_bracket_1d(Q_lin, Q_lin)
pb_PP = poisson_bracket_1d(P_lin, P_lin)

assert pb_QP == a*d - b*c
assert pb_QQ == 0
assert pb_PP == 0

J1 = Matrix([[0, 1], [-1, 0]])
M1 = Matrix([[a, b], [c, d]])
assert simplify(M1 * J1 * M1.T - (a*d - b*c) * J1) == zeros(2, 2)
assert simplify((M1 * J1 * M1.T).subs(a*d - b*c, 1) - J1) == zeros(2, 2)

# Additional 2-DOF examples: exchange and scaling transformations.
q1, q2, p1, p2, alpha = sp.symbols('q1 q2 p1 p2 alpha', real=True, nonzero=True)

M_exchange = Matrix([[0, 0, 1, 0],
                     [0, 0, 0, 1],
                     [-1, 0, 0, 0],
                     [0, -1, 0, 0]])
assert simplify(M_exchange * J * M_exchange.T - J) == zeros(4, 4)

M_scale = Matrix([[alpha, 0,     0,       0],
                  [0,     alpha, 0,       0],
                  [0,     0,     1/alpha, 0],
                  [0,     0,     0,       1/alpha]])
assert simplify(M_scale * J * M_scale.T - J) == zeros(4, 4)

print('All canonical-transformation verification checks passed.')


## Source

📖 **View this entry:** [theoria-dataset.org/entries.html?entry=canonical_transformations.json](https://theoria-dataset.org/entries.html?entry=canonical_transformations.json)

This verification code is part of the [THEORIA dataset](https://github.com/theoria-dataset/theoria-dataset), a curated collection of theoretical physics derivations with programmatic verification.

**License:** CC-BY 4.0